## 01_data_acquisition
  Checks that the downloaded raw inputs (GISCO boundaries, OSM extracts) are present and readable.
  Input: data/raw/. Output: none. Environment: nuts3-scaling.

In [ ]:
import sys
from pathlib import Path
import json

import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EUROSTAT_SOURCES

RAW_DIR = PROJECT_ROOT / "data" / "raw"
NUTS3_DIR = RAW_DIR / "nuts3"
NUTS2_DIR = RAW_DIR / "nuts2"
OSM_DIR = RAW_DIR / "osm"
EUROSTAT_DIR = RAW_DIR / "eurostat"

COUNTRY_CODES = ["hu", "pl", "cz", "sk"]

In [ ]:
def check_file(path: Path, min_bytes: int = 100) -> bool:
    ok = path.exists() and path.stat().st_size >= min_bytes
    status = "OK " if ok else "HIÁNYZIK"
    rel_path = path.relative_to(PROJECT_ROOT)
    size = path.stat().st_size if path.exists() else 0
    print(f"  [{status}] {rel_path} ({size:,} byte)")
    return ok

nuts3_path = NUTS3_DIR / "NUTS_RG_03M_2024_4326.geojson"
nuts2_path = NUTS2_DIR / "NUTS_RG_03M_2024_4326_LEVL_2.geojson"
osm_paths = {code: OSM_DIR / f"{code}-latest.osm.pbf" for code in COUNTRY_CODES}

print("NUTS3/NUTS2 határok:")
check_file(nuts3_path)
check_file(nuts2_path)

print("\nOSM .pbf fájlok:")
for code, p in osm_paths.items():
    check_file(p, min_bytes=10_000_000)

def extract_dataset_code(value):
    if isinstance(value, dict):
        return value.get("dataset_code") or value.get("code")
    return value

print("\nEurostat táblák:")
eurostat_files = sorted(EUROSTAT_DIR.glob("*.tsv"))
expected_dataset_codes = sorted({extract_dataset_code(v) for v in EUROSTAT_SOURCES.values()})
print(f"  talált .tsv fájl: {len(eurostat_files)} (várt egyedi dataset-kód: {len(expected_dataset_codes)})")
for f in eurostat_files:
    check_file(f)

found_codes = {f.stem for f in eurostat_files}
missing = set(expected_dataset_codes) - found_codes
if missing:
    print(f"  hiányzó dataset-kód(ok): {sorted(missing)}")
else:
    print("  minden várt dataset-kódhoz van letöltött fájl")

In [ ]:
nuts3_gdf = gpd.read_file(nuts3_path)
nuts2_gdf = gpd.read_file(nuts2_path)

print(f"NUTS3 CRS: {nuts3_gdf.crs}")
print(f"NUTS3 oszlopok: {list(nuts3_gdf.columns)}")
print(f"NUTS3 összes régió (egész EU): {len(nuts3_gdf)}")

print(f"\nNUTS2 CRS: {nuts2_gdf.crs}")
print(f"NUTS2 oszlopok: {list(nuts2_gdf.columns)}")
print(f"NUTS2 összes régió (egész EU): {len(nuts2_gdf)}")

v4_nuts3 = nuts3_gdf[nuts3_gdf["CNTR_CODE"].isin(["HU", "PL", "CZ", "SK"])]
v4_nuts2 = nuts2_gdf[nuts2_gdf["CNTR_CODE"].isin(["HU", "PL", "CZ", "SK"])]
print(f"\nV4 NUTS3 régiók száma: {len(v4_nuts3)}")
print(f"V4 NUTS2 régiók száma: {len(v4_nuts2)}")
print(v4_nuts3.groupby("CNTR_CODE").size())

In [ ]:
import pyrosm

bbox = [19.03, 47.47, 19.08, 47.52]  # kb. Budapest V. kerülete, [west, south, east, north]

osm = pyrosm.OSM(str(osm_paths["hu"]), bounding_box=bbox)
driving_test = osm.get_network(network_type="driving")

print(f"pyrosm verzió: {pyrosm.__version__}")
print(f"driving hálózat elemszám (teszt, bbox): {len(driving_test)}")
print(f"oszlopok: {list(driving_test.columns)}")